# Engine benchmark: four sources, three workloadsEach of four variants answers each of three identical questions. Variant is thestorage and execution path; workload is the question.| Variant | Execution | Companies source | Financial source ||---|---|---|---|| A | `mongod`, server-side C++ | MongoDB | MongoDB || B | Spark JVM, local mode | MongoDB via connector | MongoDB via connector || C | Spark JVM, local mode | Parquet | Parquet || D | Spark JVM, local mode | raw `enheter_alle.json` | NDJSON export |Variant A is not a Python join. pymongo sends the pipeline to `mongod`, whichexecutes it; Python only deserialises the result. The comparison is between adatabase engine and a distributed framework, not between languages.**Variant D is asymmetric by design.** `enheter_alle.json` is a singlepretty-printed JSON array of 1,171,373 records, 2.00 GB. Spark can only readthat with `multiLine=true`, which is not splittable: one task parses the entirefile on one thread regardless of `local[4]`. The financial side is NDJSON anddoes read in parallel. This mirrors the real difference between a bulk downloadand an incremental fetch, and the per-side read timings separate the two effectsrather than reporting one blended number.## Workloads**W1 — selective join, tiny output.** Join `financial_data` to `companies` on`organisasjonsnummer`, keep AS, count by `fetch_status`. Two output rows.**W3 — unindexed predicate, no join.** Count `konkurs = true` grouped by`organisasjonsform.kode`. There is no index on `konkurs`, so variant A must scanthe whole collection, where W1 was served through the `organisasjonsnummer`index. No join, which isolates companies-side read cost.**W4 — wide read plus real aggregation.** For AS companies with a filedstatement, aggregate operating revenue, operating profit, equity and debt byindustry code and municipality. Touches deeply nested columns across bothsources and produces tens of thousands of groups, so the bottleneck moves fromscan to shuffle. Its output feeds the analysis chapter.## Ordering**The read-only timings now run last.** In the previous version they ran first,and variant A's W1 time was 6.4x worse than an earlier run of the same pipeline(46.0s against 7.1s). Pulling 2.00 GB of JSON plus the Parquet and NDJSON filesthrough the page cache before A executes is a plausible cause, because it canevict the pages `mongod` was serving from. Moving the read-only cell to the endremoves that as a variable. This is a hypothesis, not a demonstrated cause —`Diagnose_variant_a.ipynb` tests it directly, and that notebook should be run ina fresh kernel before this one.If the diagnostic confirms cross-variant contention, set `SELECTED_VARIANTS`below to one variant at a time and run this notebook once per variant in a freshkernel. Results are written to a per-selection filename so runs do not overwriteeach other.## CorrectnessTimings from variants that disagree are meaningless, so every workload ischecked across all variants that completed. Group keys and integer counts arecompared exactly. Floating-point sums are compared with a relative tolerance,because summation order differs between `mongod` and Spark's shuffle.Two engine semantics differ and are reconciled explicitly rather than paperedover. A `$group` `_id` sub-field whose source path is missing is **omitted** fromthe MongoDB result document rather than stored as null, so those keys are readwith `.get()`. And `$sum` over an all-missing group returns `0` in MongoDB but`null` in Spark, so the Spark aggregate coalesces to `0.0`. Both cases occurhere: `naeringskode1` is absent in 35,727 companies, `kommunenummer` in 62,388,and `sumDriftsinntekter` in 91,026 filings.Run `Export_to_parquet.ipynb` and `Export_to_ndjson.ipynb` first. Restart thekernel before running this.

In [1]:
import json
import os
import statistics
import time

from pyspark.sql import SparkSession
from pyspark.sql import functions as F

from schemas import COMPANIES_SCHEMA, FINANCIAL_SCHEMA, EXPECTED_ROWS

MONGO_DB = "companiesdb"
DATA_DIR = "/home/jovyan/data"
PARQUET_DIR = os.path.join(DATA_DIR, "parquet")
NDJSON_DIR = os.path.join(DATA_DIR, "ndjson")
RAW_COMPANIES = os.path.join(DATA_DIR, "enheter_alle.json")

REPEATS = 3          # timed runs per variant, after one discarded warm-up

# Restrict to a subset to run each variant in its own kernel. Keep all four for
# a single-session run.
SELECTED_VARIANTS = ["A", "B", "C", "D"]

# Driver memory, thread count, the Mongo connector package and the connection
# URI all come from jupyter/spark-defaults.conf, which is baked into the image.
spark = SparkSession.builder.appName("group13_engine_benchmark").getOrCreate()

_conf = spark.sparkContext.getConf()
MONGO_URI = _conf.get("spark.mongodb.read.connection.uri")
CONNECTOR = _conf.get("spark.jars.packages")

print("Spark        ", spark.version)
print("master       ", spark.sparkContext.master)
print("driver heap  %.1f GB" % (spark._jvm.java.lang.Runtime.getRuntime().maxMemory() / 1024**3))
print("connector    ", CONNECTOR)
print("host cores   ", os.cpu_count())
print("variants     ", SELECTED_VARIANTS)

Spark         4.2.0
master        local[4]
driver heap  8.0 GB
connector     org.mongodb.spark:mongo-spark-connector_2.13:11.1.0
host cores    12
variants      ['A', 'B', 'C', 'D']


In [2]:
from pymongo import MongoClient

client = MongoClient(MONGO_URI)
db = client[MONGO_DB]

# These figures must appear in the report. A timing without the hardware it ran
# on is not reproducible.
with open("/proc/meminfo") as fh:
    total_kb = int(fh.readline().split()[1])

print("Container memory:    %.1f GB" % (total_kb / 1024**2))
print("Default parallelism: ", spark.sparkContext.defaultParallelism)
print("MongoDB version:     ", db.command("buildInfo")["version"])
print("Raw JSON size:       %.2f GB" % (os.path.getsize(RAW_COMPANIES) / 1000**3))

# Collection sizes are recorded because they are the first thing to check if a
# timing changes between runs.
for name in ["companies", "financial_data"]:
    n = db[name].count_documents({})
    print("%-16s %d docs  %s" % (name, n, "OK" if n == EXPECTED_ROWS[name] else "CHANGED"))


def timeit(fn, repeats=REPEATS):
    """
    Runs fn once untimed so the page cache is warm and the JVM has JIT-compiled
    the hot path, then times `repeats` further runs. Median is reported rather
    than mean, so a single GC pause or Docker scheduling hiccup does not
    dominate.

    Returns (result, times, error). On failure the error is recorded and the
    variant is reported as not completing, rather than aborting the notebook.
    """
    try:
        result = fn()
        times = []
        for _ in range(repeats):
            t0 = time.perf_counter()
            result = fn()
            times.append(time.perf_counter() - t0)
        return result, times, None
    except Exception as exc:                       # noqa: BLE001
        return None, [], "%s: %s" % (type(exc).__name__, exc)

Container memory:    15.2 GB
Default parallelism:  4
MongoDB version:      8.3.8
Raw JSON size:       2.00 GB
companies        1171373 docs  OK
financial_data   1170290 docs  OK


## Source readersEach variant's readers are defined once. Nothing is cached: every timed runre-reads from its source, which is the cost being measured.

In [3]:
def mongo_df(collection, schema):
    return (spark.read.format("mongodb")
            .option("database", MONGO_DB).option("collection", collection)
            .schema(schema).load())


def parquet_df(collection):
    return spark.read.parquet(os.path.join(PARQUET_DIR, collection))


def json_companies():
    # multiLine is mandatory: the file is one array, not one object per line.
    # It also makes the read single-threaded, which is the point of variant D.
    return (spark.read.schema(COMPANIES_SCHEMA)
            .option("multiLine", "true").json(RAW_COMPANIES))


def json_financial():
    return (spark.read.schema(FINANCIAL_SCHEMA)
            .json(os.path.join(NDJSON_DIR, "financial_data")))


ALL_SOURCES = {
    "B: Spark + Mongo connector": (lambda: mongo_df("companies", COMPANIES_SCHEMA),
                                   lambda: mongo_df("financial_data", FINANCIAL_SCHEMA)),
    "C: Spark + Parquet":         (lambda: parquet_df("companies"),
                                   lambda: parquet_df("financial_data")),
    "D: Spark + JSON":            (json_companies, json_financial),
}

SOURCES = {k: v for k, v in ALL_SOURCES.items() if k[0] in SELECTED_VARIANTS}
RUN_MONGO = "A" in SELECTED_VARIANTS
print("Spark variants:", list(SOURCES))
print("MongoDB variant:", RUN_MONGO)

Spark variants: ['B: Spark + Mongo connector', 'C: Spark + Parquet', 'D: Spark + JSON']
MongoDB variant: True


## Result comparisonGroup keys and integer counts must match exactly across variants. Sums offloating-point columns are compared with a relative tolerance of 1e-9, because`mongod` and Spark accumulate in different orders.

In [4]:
TOLERANCE = 1e-9


def normalise(rows):
    """
    rows: iterable of (key_tuple, int_count, tuple_of_floats).
    Sorted by key so variants can be compared elementwise. A missing key and an
    empty string are folded together, because MongoDB omits missing _id
    sub-fields while Spark carries them as null.
    """
    out = []
    for key, count, floats in rows:
        key = tuple("" if k is None else str(k) for k in key)
        floats = tuple(None if f is None else float(f) for f in floats)
        out.append((key, int(count), floats))
    return sorted(out)


def compare(a, b):
    """True if two normalised results agree within tolerance."""
    if a is None or b is None or len(a) != len(b):
        return False
    for (ka, ca, fa), (kb, cb, fb) in zip(a, b):
        if ka != kb or ca != cb or len(fa) != len(fb):
            return False
        for x, y in zip(fa, fb):
            if (x is None) != (y is None):
                return False
            if x is None:
                continue
            scale = max(abs(x), abs(y), 1.0)
            if abs(x - y) / scale > TOLERANCE:
                return False
    return True


def report(wdata):
    for label, (res, times, err) in wdata.items():
        if err:
            print("%-28s DID NOT COMPLETE  %s" % (label, err))
        else:
            print("%-28s median %6.2fs   min %6.2fs   max %6.2fs   %d rows"
                  % (label, statistics.median(times), min(times), max(times), len(res)))

## W1 — selective join, two output rows

In [5]:
W1_PIPELINE = [
    {"$lookup": {"from": "companies", "localField": "organisasjonsnummer",
                 "foreignField": "organisasjonsnummer", "as": "company"}},
    {"$unwind": "$company"},
    {"$match": {"company.organisasjonsform.kode": "AS"}},
    {"$group": {"_id": "$fetch_status", "count": {"$sum": 1}}},
]


def w1_mongo():
    rows = db.financial_data.aggregate(W1_PIPELINE, allowDiskUse=True)
    return normalise(((r["_id"],), r["count"], ()) for r in rows)


def w1_spark(comp_fn, fin_fn):
    companies = comp_fn().filter("organisasjonsform.kode = 'AS'").select("organisasjonsnummer")
    financial = fin_fn().select("organisasjonsnummer", "fetch_status")
    rows = (financial.join(companies, "organisasjonsnummer")
            .groupBy("fetch_status").count().collect())
    return normalise(((r["fetch_status"],), r["count"], ()) for r in rows)


w1 = {}
if RUN_MONGO:
    w1["A: MongoDB aggregation"] = timeit(w1_mongo)
for label, (comp_fn, fin_fn) in SOURCES.items():
    w1[label] = timeit(lambda c=comp_fn, f=fin_fn: w1_spark(c, f))

report(w1)

A: MongoDB aggregation       median  40.28s   min  39.80s   max  42.70s   2 rows
B: Spark + Mongo connector   median   9.07s   min   8.88s   max   9.30s   2 rows
C: Spark + Parquet           median   2.49s   min   2.40s   max   3.09s   2 rows
D: Spark + JSON              median  36.93s   min  36.78s   max  37.00s   2 rows


## W3 — unindexed predicate, no join`konkurs` carries no index, so variant A scans the full collection here ratherthan seeking through `organisasjonsnummer` as it did in W1.

In [6]:
W3_PIPELINE = [
    {"$match": {"konkurs": True}},
    {"$group": {"_id": "$organisasjonsform.kode", "count": {"$sum": 1}}},
]


def w3_mongo():
    rows = db.companies.aggregate(W3_PIPELINE, allowDiskUse=True)
    return normalise(((r["_id"],), r["count"], ()) for r in rows)


def w3_spark(comp_fn):
    rows = (comp_fn().filter(F.col("konkurs"))
            .groupBy(F.col("organisasjonsform.kode").alias("kode"))
            .count().collect())
    return normalise(((r["kode"],), r["count"], ()) for r in rows)


w3 = {}
if RUN_MONGO:
    w3["A: MongoDB aggregation"] = timeit(w3_mongo)
for label, (comp_fn, _) in SOURCES.items():
    w3[label] = timeit(lambda c=comp_fn: w3_spark(c))

report(w3)

A: MongoDB aggregation       median   0.35s   min   0.35s   max   0.37s   11 rows
B: Spark + Mongo connector   median   1.35s   min   1.34s   max   1.40s   11 rows
C: Spark + Parquet           median   1.49s   min   1.39s   max   1.76s   11 rows
D: Spark + JSON              median  32.07s   min  31.31s   max  37.05s   11 rows


## W4 — wide read plus real aggregationOperating revenue, operating profit, equity and debt for AS companies with afiled statement, grouped by industry code and municipality.`data` is typed as an array because that is what the API returns, but theprofiling pass confirmed exactly one element in all 444,644 populated records,so element 0 is taken directly and the join stays one-to-one. Variant A uses`$unwind`, which is equivalent at multiplicity 1.Two engine differences are handled explicitly here. MongoDB omits a `_id`sub-field whose source path is missing rather than storing null, so the groupkeys are read with `.get()`. And `$sum` over an all-missing group returns `0` inMongoDB but `null` in Spark, so the Spark aggregate coalesces to `0.0`. Withoutboth, the variants would report a spurious disagreement.

In [7]:
W4_PIPELINE = [
    {"$match": {"fetch_status": "success"}},
    {"$unwind": "$data"},
    {"$lookup": {"from": "companies", "localField": "organisasjonsnummer",
                 "foreignField": "organisasjonsnummer", "as": "c"}},
    {"$unwind": "$c"},
    {"$match": {"c.organisasjonsform.kode": "AS"}},
    {"$group": {
        "_id": {"naering": "$c.naeringskode1.kode",
                "kommune": "$c.forretningsadresse.kommunenummer"},
        "n": {"$sum": 1},
        "inntekt": {"$sum": "$data.resultatregnskapResultat.driftsresultat.driftsinntekter.sumDriftsinntekter"},
        "driftsres": {"$sum": "$data.resultatregnskapResultat.driftsresultat.driftsresultat"},
        "egenkapital": {"$sum": "$data.egenkapitalGjeld.egenkapital.sumEgenkapital"},
        "gjeld": {"$sum": "$data.egenkapitalGjeld.gjeldOversikt.sumGjeld"},
    }},
]

W4_SUMS = ["inntekt", "driftsres", "egenkapital", "gjeld"]


def w4_mongo():
    rows = db.financial_data.aggregate(W4_PIPELINE, allowDiskUse=True)
    # A missing source path is omitted from the _id subdocument rather than
    # stored as null, so these must be .get() and not subscripts.
    # naeringskode1 is absent in 35,727 companies, kommunenummer in 62,388.
    return normalise(
        ((r["_id"].get("naering"), r["_id"].get("kommune")), r["n"],
         tuple(r[k] for k in W4_SUMS))
        for r in rows)


def w4_spark(comp_fn, fin_fn):
    companies = comp_fn().filter("organisasjonsform.kode = 'AS'").select(
        "organisasjonsnummer",
        F.col("naeringskode1.kode").alias("naering"),
        F.col("forretningsadresse.kommunenummer").alias("kommune"))
    financial = fin_fn().filter("fetch_status = 'success'").select(
        "organisasjonsnummer", F.col("data")[0].alias("d"))

    # Mongo's $sum yields 0 for a group where every value is missing; Spark's
    # sum() yields null. sumDriftsinntekter is absent in 91,026 filings, so
    # all-null groups do occur.
    def s(path, name):
        return F.coalesce(F.sum(path), F.lit(0.0)).alias(name)

    rows = (financial.join(companies, "organisasjonsnummer")
            .groupBy("naering", "kommune")
            .agg(F.count(F.lit(1)).alias("n"),
                 s("d.resultatregnskapResultat.driftsresultat.driftsinntekter.sumDriftsinntekter", "inntekt"),
                 s("d.resultatregnskapResultat.driftsresultat.driftsresultat", "driftsres"),
                 s("d.egenkapitalGjeld.egenkapital.sumEgenkapital", "egenkapital"),
                 s("d.egenkapitalGjeld.gjeldOversikt.sumGjeld", "gjeld"))
            .collect())
    return normalise(((r["naering"], r["kommune"]), r["n"],
                      tuple(r[k] for k in W4_SUMS)) for r in rows)


w4 = {}
if RUN_MONGO:
    w4["A: MongoDB aggregation"] = timeit(w4_mongo)
for label, (comp_fn, fin_fn) in SOURCES.items():
    w4[label] = timeit(lambda c=comp_fn, f=fin_fn: w4_spark(c, f))

report(w4)

A: MongoDB aggregation       median  18.31s   min  18.23s   max  18.46s   45033 rows
B: Spark + Mongo connector   median  19.66s   min  16.63s   max  20.25s   45033 rows
C: Spark + Parquet           median   3.15s   min   2.98s   max   3.33s   45033 rows
D: Spark + JSON              median  43.68s   min  42.33s   max  44.30s   45033 rows


## CorrectnessThe reference is the first variant that completed, not variant Aunconditionally. In the previous version A was hardcoded as reference, so whenA failed on W4 every other variant was reported as disagreeing when in factthey had not been compared to anything.

In [8]:
WORKLOADS = [("W1", w1), ("W3", w3), ("W4", w4)]

agreement = {}
for wname, wdata in WORKLOADS:
    completed = [(l, r) for l, (r, t, e) in wdata.items() if not e]
    agreement[wname] = {}
    print("\n=== %s ===" % wname)
    if not completed:
        print("  no variant completed")
        continue
    ref_label, ref = completed[0]
    agreement[wname]["_reference"] = ref_label
    print("  reference: %s" % ref_label)
    for label, (res, times, err) in wdata.items():
        if err:
            print("  %-28s did not complete" % label)
            continue
        ok = compare(ref, res)
        agreement[wname][label] = ok
        print("  %-28s %s  (%d rows)" % (label, "agrees" if ok else "DISAGREES", len(res)))


=== W1 ===
  reference: A: MongoDB aggregation
  A: MongoDB aggregation       agrees  (2 rows)
  B: Spark + Mongo connector   agrees  (2 rows)
  C: Spark + Parquet           agrees  (2 rows)
  D: Spark + JSON              agrees  (2 rows)

=== W3 ===
  reference: A: MongoDB aggregation
  A: MongoDB aggregation       agrees  (11 rows)
  B: Spark + Mongo connector   agrees  (11 rows)
  C: Spark + Parquet           agrees  (11 rows)
  D: Spark + JSON              agrees  (11 rows)

=== W4 ===
  reference: A: MongoDB aggregation
  A: MongoDB aggregation       agrees  (45033 rows)
  B: Spark + Mongo connector   agrees  (45033 rows)
  C: Spark + Parquet           agrees  (45033 rows)
  D: Spark + JSON              agrees  (45033 rows)


## Read-only timingsDeliberately last. These pull the entire 2.00 GB JSON file plus the Parquet andNDJSON exports through the page cache, which can evict the pages `mongod` isserving from and penalise any MongoDB variant that runs afterwards. Running themat the end keeps that effect out of the workload timings.This is where the array-versus-NDJSON contrast inside variant D is visible: sameengine, same machine, same session, with file framing as the only variable.

In [9]:
read_times = {}
for label, (comp_fn, fin_fn) in SOURCES.items():
    for side, fn in [("companies", comp_fn), ("financial_data", fin_fn)]:
        key = "%s | %s" % (label, side)
        _, times, err = timeit(lambda fn=fn: fn().count(), repeats=1)
        read_times[key] = {"times": times, "error": err}
        if err:
            print("%-44s FAILED  %s" % (key, err))
        else:
            print("%-44s %7.2fs" % (key, times[0]))

B: Spark + Mongo connector | companies          8.09s
B: Spark + Mongo connector | financial_data     4.59s
C: Spark + Parquet | companies                  1.60s
C: Spark + Parquet | financial_data             0.74s
D: Spark + JSON | companies                    31.31s
D: Spark + JSON | financial_data                4.24s


## Summary and persistence

In [10]:
print("%-10s %-28s %9s %9s %9s %9s" % ("workload", "variant", "median", "min", "max", "vs best"))
print("-" * 82)
for wname, wdata in WORKLOADS:
    completed = {l: t for l, (r, t, e) in wdata.items() if not e}
    if not completed:
        continue
    best = min(statistics.median(t) for t in completed.values())
    for label, (res, times, err) in wdata.items():
        if err:
            print("%-10s %-28s %9s" % (wname, label, "FAILED"))
            continue
        med = statistics.median(times)
        print("%-10s %-28s %8.2fs %8.2fs %8.2fs %8.2fx"
              % (wname, label, med, min(times), max(times), med / best))

summary = {
    "repeats": REPEATS,
    "selected_variants": SELECTED_VARIANTS,
    "read_only_timings_ran": "last",
    "cpu_cores": os.cpu_count(),
    "spark_master": spark.sparkContext.master,
    "driver_max_heap_gb": spark._jvm.java.lang.Runtime.getRuntime().maxMemory() / 1024**3,
    "spark_version": spark.version,
    "mongodb_version": db.command("buildInfo")["version"],
    "connector": CONNECTOR,
    "raw_json_bytes": os.path.getsize(RAW_COMPANIES),
    "collection_counts": {n: db[n].count_documents({}) for n in ["companies", "financial_data"]},
    "read_only_seconds": read_times,
    "workloads": {
        wname: {label: {"times": times, "error": err,
                        "rows": None if res is None else len(res)}
                for label, (res, times, err) in wdata.items()}
        for wname, wdata in WORKLOADS
    },
    "agreement": agreement,
    "float_tolerance": TOLERANCE,
}

# Per-selection filename so a partial run does not overwrite a full one.
suffix = "" if SELECTED_VARIANTS == ["A", "B", "C", "D"] else "_" + "".join(SELECTED_VARIANTS)
out_path = os.path.join(DATA_DIR, "benchmark_results%s.json" % suffix)
with open(out_path, "w") as fh:
    json.dump(summary, fh, indent=2)
print("\nSaved to %s" % out_path)

# W4's aggregate is the input to the analysis chapter, so it is kept rather
# than discarded with the timings. Prefer Parquet as the source since it is the
# fastest variant that completed; fall back to whichever did.
w4_result = next((r for l, (r, t, e) in w4.items() if not e and l.startswith("C")), None)
if w4_result is None:
    w4_result = next((r for l, (r, t, e) in w4.items() if not e), None)
if w4_result:
    with open(os.path.join(DATA_DIR, "w4_industry_municipality.json"), "w") as fh:
        json.dump([{"naeringskode": k[0] or None, "kommunenummer": k[1] or None,
                    "antall": n, "sumDriftsinntekter": f[0], "sumDriftsresultat": f[1],
                    "sumEgenkapital": f[2], "sumGjeld": f[3]}
                   for k, n, f in w4_result], fh, indent=1)
    print("Saved %d groups to data/w4_industry_municipality.json" % len(w4_result))

workload   variant                         median       min       max   vs best
----------------------------------------------------------------------------------
W1         A: MongoDB aggregation          40.28s    39.80s    42.70s    16.20x
W1         B: Spark + Mongo connector       9.07s     8.88s     9.30s     3.65x
W1         C: Spark + Parquet               2.49s     2.40s     3.09s     1.00x
W1         D: Spark + JSON                 36.93s    36.78s    37.00s    14.85x
W3         A: MongoDB aggregation           0.35s     0.35s     0.37s     1.00x
W3         B: Spark + Mongo connector       1.35s     1.34s     1.40s     3.83x
W3         C: Spark + Parquet               1.49s     1.39s     1.76s     4.23x
W3         D: Spark + JSON                 32.07s    31.31s    37.05s    90.82x
W4         A: MongoDB aggregation          18.31s    18.23s    18.46s     5.81x
W4         B: Spark + Mongo connector      19.66s    16.63s    20.25s     6.24x
W4         C: Spark + Parquet        